# Cross-Encoder Fine-Tuning v0.6 — Experiment Notebook

**Goal**: Break the 60.76% test LabelAcc ceiling with a balanced, larger dataset (13,350 pairs with perfect 20%/20%/20%/20%/20% distribution).

**Hypothesis**: Class balance + 91% more data eliminates MSELoss's pull toward poor_match; baseline MSELoss+Spearman should exceed v0.2's ceiling.

**Dataset**: v0.5 (13,350 pairs)
- Train: 9,350 pairs
- Validation: 2,000 pairs (doubled from 1,050 → more stable checkpoint selection)
- Test: 2,000 pairs

**Three core runs**:
1. MSELoss + Spearman, 10 epochs (baseline)
2. BoundaryAwareLoss + Spearman, 10 epochs (test if helps on balanced data)
3. MSELoss + LabelAcc, 10 epochs (re-test evaluator with 2x val set)


## Setup


In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# Clone or sync the repository
import os
import subprocess

repo_path = '/content/gdrive/My Drive/Ai-Recruiter-Mini-Ai-Service'

if not os.path.exists(repo_path):
    print("Cloning repository...")
    os.chdir('/content/gdrive/My Drive')
    subprocess.run(['git', 'clone', 'https://github.com/DangHuuLong/Ai-Recruiter-Mini-Ai-Service.git'], check=True)
else:
    print(f"Repository already exists at {repo_path}")
    print("Pulling latest changes...")
    os.chdir(repo_path)
    subprocess.run(['git', 'pull', 'origin', 'experiment/cross-encoder-v0.6'], check=False)

os.chdir(repo_path)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies from requirements.txt
!pip install -q -r requirements.txt

In [ ]:
# Verify dataset files exist
import os
dataset_path = 'datasets/versions/v0.5/cross_encoder'
files = os.listdir(dataset_path) if os.path.exists(dataset_path) else []
print(f"Files in {dataset_path}:")
for f in sorted(files):
    fpath = os.path.join(dataset_path, f)
    size_mb = os.path.getsize(fpath) / 1e6
    print(f"  {f}: {size_mb:.1f} MB")

## Helper Functions


In [ ]:
import json
import math
import torch
import torch.nn.functional as F
import numpy as np
from pathlib import Path
from typing import Any
from scipy.stats import spearmanr
from sentence_transformers import CrossEncoder, InputExample

# Loss function: BoundaryAwareLoss
class BoundaryAwareLoss(torch.nn.Module):
    """MSE + ordinal BCE at boundaries (40/60/75/90)."""
    _BOUNDARIES = [0.40, 0.60, 0.75, 0.90]
    _ALPHA = 0.3

    def forward(self, preds: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        mse = F.mse_loss(preds, labels)
        ordinal = preds.new_zeros(1)
        for b in self._BOUNDARIES:
            target = (labels >= b).float()
            pred_logit = 20.0 * (preds - b)
            ordinal = ordinal + F.binary_cross_entropy_with_logits(pred_logit, target)
        return mse + self._ALPHA * ordinal / len(self._BOUNDARIES)

# Evaluator: CECorrelationEvaluator (Spearman)
class CECorrelationEvaluator:
    """Evaluate cross-encoder using Spearman correlation."""
    def __init__(self, sentence_pairs: list, labels_0_1: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_1 = labels_0_1
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CECorrelationEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_1=[ex.label for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        corr, _ = spearmanr(preds, self.labels_0_1)
        return float(corr) if not np.isnan(corr) else 0.0

# Evaluator: CELabelAccEvaluator
class CELabelAccEvaluator:
    def __init__(self, sentence_pairs: list, labels_0_100: list[float], name: str = ""):
        self.sentence_pairs = sentence_pairs
        self.labels_0_100 = labels_0_100
        self.name = name

    @classmethod
    def from_input_examples(cls, examples, name: str = "") -> "CELabelAccEvaluator":
        return cls(
            sentence_pairs=[ex.texts for ex in examples],
            labels_0_100=[ex.label * 100 for ex in examples],
            name=name,
        )

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1) -> float:
        preds = model.predict(self.sentence_pairs, batch_size=32, show_progress_bar=False)
        arr = np.asarray(preds)
        if arr.ndim == 2:
            # Classification mode (5-class)
            pred_classes = arr.argmax(axis=1).tolist()
            true_classes = [_score_to_class_index(t) for t in self.labels_0_100]
            return sum(1 for p, t in zip(pred_classes, true_classes) if p == t) / len(pred_classes)
        # Regression mode
        pred_100 = [float(p) * 100 for p in preds]
        label_acc = sum(
            1 for p, t in zip(pred_100, self.labels_0_100)
            if abs(p - t) <= 10  # Within ±10 points → correct label
        ) / len(pred_100)
        return label_acc

def _score_to_class_index(score: float) -> int:
    """Convert 0-100 score to 5-class index."""
    if score < 40:
        return 0  # poor
    elif score < 60:
        return 1  # weak
    elif score < 75:
        return 2  # moderate
    elif score < 90:
        return 3  # strong
    else:
        return 4  # excellent

def load_jsonl(path: str) -> list[dict[str, Any]]:
    """Load JSONL file."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def load_dataset(data_dir: str) -> tuple[list[InputExample], list[InputExample], list[InputExample]]:
    """Load train/val/test InputExample lists."""
    train_data = load_jsonl(f"{data_dir}/cross_encoder_train.jsonl")
    val_data = load_jsonl(f"{data_dir}/cross_encoder_validation.jsonl")
    test_data = load_jsonl(f"{data_dir}/cross_encoder_test.jsonl")
    
    def to_input_examples(records):
        return [
            InputExample(texts=[rec['cv_text'], rec['jd_text']], label=rec['label'])
            for rec in records
        ]
    
    return to_input_examples(train_data), to_input_examples(val_data), to_input_examples(test_data)

def compute_metrics(model: CrossEncoder, examples: list[InputExample], batch_size: int = 32) -> dict:
    """Compute MAE, RMSE, LabelAcc on examples."""
    preds = model.predict([ex.texts for ex in examples], batch_size=batch_size, show_progress_bar=False)
    preds_100 = np.asarray(preds) * 100
    labels_100 = np.array([ex.label * 100 for ex in examples])
    
    mae = np.mean(np.abs(preds_100 - labels_100))
    rmse = np.sqrt(np.mean((preds_100 - labels_100) ** 2))
    label_acc = np.mean(np.abs(preds_100 - labels_100) <= 10)
    
    return {'MAE': mae, 'RMSE': rmse, 'LabelAcc': label_acc}

print("Helper functions loaded.")

## Load Dataset


In [ ]:
train_examples, val_examples, test_examples = load_dataset('datasets/versions/v0.5/cross_encoder')

print(f"Train: {len(train_examples)} pairs")
print(f"Val:   {len(val_examples)} pairs")
print(f"Test:  {len(test_examples)} pairs")
print(f"Total: {len(train_examples) + len(val_examples) + len(test_examples)} pairs")

## Run 1: MSELoss + Spearman, 10 epochs (Baseline)


In [ ]:
import torch

run_name = "v0.6-run1-mse-spearman"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

# Create model
model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

# Spearman evaluator (checkpoint selection)
evaluator = CECorrelationEvaluator.from_input_examples(
    val_examples,
    name="val_spearman"
)

# Train
print(f"\n🚀 Starting {run_name}...")
model.fit(
    train_objectives=[(train_examples, torch.nn.MSELoss())],
    evaluator=evaluator,
    epochs=10,
    batch_size=16,
    warmup_ratio=0.1,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

# Evaluate best model
best_model_path = output_dir
best_model = CrossEncoder(best_model_path)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

# Save results
results_run1 = {
    'run': run_name,
    'loss': 'MSE',
    'evaluator': 'Spearman',
    'epochs': 10,
    'batch_size': 16,
    'val': val_metrics,
    'test': test_metrics
}

with open(f'artifacts/reports/{run_name}_results.json', 'w') as f:
    json.dump(results_run1, f, indent=2)

print(f"\nResults saved to artifacts/reports/{run_name}_results.json")

## Run 2: BoundaryAwareLoss + Spearman, 10 epochs


In [ ]:
import torch

run_name = "v0.6-run2-boundary-spearman"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

# Create model
model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

# Spearman evaluator
evaluator = CECorrelationEvaluator.from_input_examples(
    val_examples,
    name="val_spearman"
)

# BoundaryAwareLoss
boundary_loss = BoundaryAwareLoss()

# Train
print(f"\n🚀 Starting {run_name}...")
model.fit(
    train_objectives=[(train_examples, boundary_loss)],
    evaluator=evaluator,
    epochs=10,
    batch_size=16,
    warmup_ratio=0.1,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

# Evaluate best model
best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

# Save results
results_run2 = {
    'run': run_name,
    'loss': 'BoundaryAwareLoss',
    'evaluator': 'Spearman',
    'epochs': 10,
    'batch_size': 16,
    'val': val_metrics,
    'test': test_metrics
}

with open(f'artifacts/reports/{run_name}_results.json', 'w') as f:
    json.dump(results_run2, f, indent=2)

print(f"\nResults saved to artifacts/reports/{run_name}_results.json")

## Run 3: MSELoss + LabelAcc, 10 epochs


In [ ]:
import torch

run_name = "v0.6-run3-mse-labelacc"
output_dir = f"artifacts/models/cross-encoder-cv-jd-{run_name}"

# Create model
model = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-12-v2',
    num_labels=1,
    default_activation_function=torch.nn.Sigmoid()
)

# LabelAcc evaluator
evaluator = CELabelAccEvaluator.from_input_examples(
    val_examples,
    name="val_label_acc"
)

# Train
print(f"\n🚀 Starting {run_name}...")
model.fit(
    train_objectives=[(train_examples, torch.nn.MSELoss())],
    evaluator=evaluator,
    epochs=10,
    batch_size=16,
    warmup_ratio=0.1,
    output_path=output_dir,
    save_best_model=True,
    use_amp=True,
    max_grad_norm=1.0,
    show_progress_bar=True
)

# Evaluate best model
best_model = CrossEncoder(output_dir)
val_metrics = compute_metrics(best_model, val_examples)
test_metrics = compute_metrics(best_model, test_examples)

print(f"\n✅ {run_name} completed.")
print(f"\nValidation metrics:")
for key, val in val_metrics.items():
    print(f"  {key}: {val:.4f}")
print(f"\nTest metrics:")
for key, val in test_metrics.items():
    print(f"  {key}: {val:.4f}")

# Save results
results_run3 = {
    'run': run_name,
    'loss': 'MSE',
    'evaluator': 'LabelAcc',
    'epochs': 10,
    'batch_size': 16,
    'val': val_metrics,
    'test': test_metrics
}

with open(f'artifacts/reports/{run_name}_results.json', 'w') as f:
    json.dump(results_run3, f, indent=2)

print(f"\nResults saved to artifacts/reports/{run_name}_results.json")

## Summary and Comparison


In [ ]:
import pandas as pd

# Compile results
summary = pd.DataFrame([
    {
        'Run': 'Run 1 (MSE+Spearman)',
        'Val LabelAcc': results_run1['val']['LabelAcc'],
        'Test LabelAcc': results_run1['test']['LabelAcc'],
        'Test MAE': results_run1['test']['MAE'],
        'Test RMSE': results_run1['test']['RMSE'],
        'Val-Test Gap (pp)': (results_run1['val']['LabelAcc'] - results_run1['test']['LabelAcc']) * 100
    },
    {
        'Run': 'Run 2 (Boundary+Spearman)',
        'Val LabelAcc': results_run2['val']['LabelAcc'],
        'Test LabelAcc': results_run2['test']['LabelAcc'],
        'Test MAE': results_run2['test']['MAE'],
        'Test RMSE': results_run2['test']['RMSE'],
        'Val-Test Gap (pp)': (results_run2['val']['LabelAcc'] - results_run2['test']['LabelAcc']) * 100
    },
    {
        'Run': 'Run 3 (MSE+LabelAcc)',
        'Val LabelAcc': results_run3['val']['LabelAcc'],
        'Test LabelAcc': results_run3['test']['LabelAcc'],
        'Test MAE': results_run3['test']['MAE'],
        'Test RMSE': results_run3['test']['RMSE'],
        'Val-Test Gap (pp)': (results_run3['val']['LabelAcc'] - results_run3['test']['LabelAcc']) * 100
    }
])

print("\n📊 Experiment Summary (v0.6 vs v0.2 baseline: 60.76%)\n")
print(summary.to_string(index=False))

# Find best run
best_idx = summary['Test LabelAcc'].idxmax()
best_run = summary.loc[best_idx]
print(f"\n🏆 Best run: {best_run['Run']} with {best_run['Test LabelAcc']:.2%} test LabelAcc")
print(f"  Ceiling broken: {'✅ YES' if best_run['Test LabelAcc'] > 0.6076 else '❌ NO'}")
print(f"  Improvement over v0.2: {(best_run['Test LabelAcc'] - 0.6076) * 100:+.2f} pp")

## Save Summary Report


In [ ]:
# Create consolidated report
report = {
    'experiment': 'cross-encoder-v0.6',
    'dataset': 'v0.5',
    'dataset_size': {
        'train': len(train_examples),
        'val': len(val_examples),
        'test': len(test_examples),
        'total': len(train_examples) + len(val_examples) + len(test_examples)
    },
    'base_model': 'cross-encoder/ms-marco-MiniLM-L-12-v2',
    'baseline_v0_2': {
        'dataset_size': 7000,
        'test_label_acc': 0.6076,
        'loss': 'MSE',
        'evaluator': 'Spearman',
        'epochs': 10
    },
    'runs': [
        {
            'name': 'Run 1: MSELoss + Spearman',
            'loss': 'MSE',
            'evaluator': 'Spearman',
            'epochs': 10,
            'val': results_run1['val'],
            'test': results_run1['test']
        },
        {
            'name': 'Run 2: BoundaryAwareLoss + Spearman',
            'loss': 'BoundaryAwareLoss (α=0.3)',
            'evaluator': 'Spearman',
            'epochs': 10,
            'val': results_run2['val'],
            'test': results_run2['test']
        },
        {
            'name': 'Run 3: MSELoss + LabelAcc',
            'loss': 'MSE',
            'evaluator': 'LabelAcc',
            'epochs': 10,
            'val': results_run3['val'],
            'test': results_run3['test']
        }
    ],
    'findings': {
        'best_test_label_acc': float(summary['Test LabelAcc'].max()),
        'best_run': summary.loc[summary['Test LabelAcc'].idxmax(), 'Run'],
        'ceiling_broken': float(summary['Test LabelAcc'].max()) > 0.6076,
        'improvement_over_v0_2_pp': (float(summary['Test LabelAcc'].max()) - 0.6076) * 100
    }
}

# Save to file
with open('artifacts/reports/v0.6_complete_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("\n✅ Complete report saved to artifacts/reports/v0.6_complete_report.json")

## Next Steps

### If ceiling broken (Test LabelAcc > 60.76%):
1. Create `docs/similarity-model-cross-encoder-v0.6.md` documenting the experiment
2. Test best model on full pipeline (bi-encoder + rerank)
3. Compare v0.6 vs v0.2 on production test cases

### If ceiling NOT broken:
1. Investigate score distribution gap at 70–79 range (only 124 pairs)
2. Generate additional moderate_match/strong_match pairs in this range
3. Re-train on expanded dataset (v0.5.1)
4. Alternatively: test larger base model (`L-12-v2` vs `L-6-v2` upgrade already done)
